# Superstore Project 
Dataset containing Sales & Profits of a Superstore

-- Business Statistics & Insights | Master in Business Analytics & AI

The goal is to use the Superstore Sales dataset (9,994 transactions, 2014–2017) not just to complete the assignment, but to produce work that directly maps to the five skill areas in the target role. Each section below defines the analysis, the method, the business framing, and the free tool to use.

* Dataset: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

### Task 4 - Customer Segmentation

Raw transaction data cannot be fed directly into a clustering algorithm because each customer appears across multiple rows. RFM (Recency, Frequency, Monetary) collapses all transactions per customer into three dimensions that capture the most predictive aspects of customer behaviour. This is the industry standard approach for customer segmentation in retail and B2B analytics.

##### A. Import libraries and dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet('../data/processed/superstore_clean.parquet')
print(f"{df.shape[0]} rows loaded")

9994 rows loaded


#### 4.1. RFM Table (Recency, Frequency, Monetary)

* RFM segmentation is a behavioral data analysis method used by marketers to group customers based on their past purchase history. 
* It evaluates three key metrics to determine which customers are the most valuable, engaged, or at risk of churning:
    * Recency: How recently a customer made a purchase.
    * Frequency: How often a customer makes a purchase.
    * Monetary Value: How much money the customer spends

In [2]:
# Reference date: one day after the last order in the dataset
reference_date = df['Order Date'].max() + pd.Timedelta(days=1)
print(f"Reference date: {reference_date.date()}")

rfm = df.groupby('Customer ID').agg(
    Customer_Name = ('Customer Name', 'first'),
    Segment       = ('Segment', 'first'),          # keep for comparison later
    Region        = ('Region', 'first'),
    Recency       = ('Order Date', lambda x: (reference_date - x.max()).days),
    Frequency     = ('Order ID', 'nunique'),
    Monetary      = ('Profit', 'sum')              # total profit generated
).reset_index()

print(f"\nCustomers: {len(rfm)}")
print(rfm.describe().round(2))

Reference date: 2017-12-31

Customers: 793
       Recency  Frequency  Monetary
count   793.00     793.00    793.00
mean    147.80       6.32    361.16
std     186.21       2.55    894.26
min       1.00       1.00  -6626.39
25%      31.00       5.00     36.61
50%      76.00       6.00    227.83
75%     184.00       8.00    560.01
max    1166.00      17.00   8981.32


Why Profit for Monetary and not Sales? 

Monetary value is defined as total profit generated rather than total revenue. A customer spending $10,000 with 40% discounts may generate less profit than one spending $3,000 at full price. Profit-based CLV is more strategically relevant for resource allocation decisions.

#### 4.2. RFM Scores (1-5)

In [3]:
# Score each dimension: higher score = better customer
# Recency: LOWER days = BETTER = higher score (hence reverse ranking)
rfm['R_Score'] = pd.qcut(rfm['Recency'],  q=5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Monetary can have ties — handle with rank
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'), q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# Composite RFM score
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print("RFM Score distribution:")
print(rfm['RFM_Score'].describe().round(2))
print(f"\nScore range: {rfm['RFM_Score'].min()} to {rfm['RFM_Score'].max()}")

RFM Score distribution:
count    793.00
mean       9.01
std        3.01
min        3.00
25%        7.00
50%        9.00
75%       11.00
max       15.00
Name: RFM_Score, dtype: float64

Score range: 3 to 15


#### 4.3. K-Means Clustering

##### 4.3.1. First: Scale the features

In [8]:
# K-means is distance-based-unscaled features will be dominated by Monetary

features = rfm[['Recency', 'Frequency', 'Monetary']].copy()

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

print("After scaling:")
print(pd.DataFrame(features_scaled, columns=['Recency', 'Frequency', 'Monetary']).describe().round(3))

After scaling:
       Recency  Frequency  Monetary
count  793.000    793.000   793.000
mean     0.000      0.000    -0.000
std      1.001      1.001     1.001
min     -0.789     -2.086    -7.819
25%     -0.628     -0.516    -0.363
50%     -0.386     -0.124    -0.149
75%      0.195      0.660     0.223
max      5.471      4.191     9.646


K-means computes Euclidean distances between points. Without scaling, Monetary (range: -$X to $X) would dominate Recency (range: 0–1400 days) and Frequency (range: 1–28 orders) simply due to its larger numerical scale. StandardScaler transforms each variable to mean=0 and std=1, giving all three equal influence.

#### 4.4. Profile each cluster: Champions, Lost etc

#### 4.5. Visualize the clusters

#### 4.6. Compare clusters to predefined segments

#### 4.7. CLV estimation per cluster - Customer Lifetime Value

* It is a metric that estimates the total net profit or revenue a business can expect from a single customer throughout their entire relationship with the company.
* Understanding CLV helps you shift focus from one-time transactions to long-term profitability. It allows you to:
    * Optimize Acquisition: Helps calculate how much you can afford to spend on marketing to acquire a new customer (known as CAC). 
    * A standard business benchmark is a CLV to CAC ratio of 3:1.
    * Improve Retention: Identifies your most valuable customer segments so you can prioritize loyalty programs, upselling, and customer service resources.
    * Forecast Revenue: Provides a lens into long-term business health and expected future cash flows.

#### 4.8. Marketing KPIs

In [ ]:
""" kpis_marketing = pd.DataFrame({
    "KPI": ["Unique Customers",
            "New Customers 2025",
            "Retention Rate",
            "Churn Rate",
            "Repeat Purchase Rate",
            "Champions Share",
            "At-Risk Share",
            "Avg CLV (all)",
            "Avg CLV (Champions)"],
    
    "Value": [
        f"{customer_analytics['Customer ID'].nunique():,}",                                                 # total unique customers across all years
        f"{len(active_2025 - active_2024):,}",                                                              # new customers in 2025 who were not active in 2024
        f"{len(retained)/len(active_2024):.1%}",                                                            # retention rate calculated as the proportion of customers active in 2024 who were retained in 2025
        f"{1-len(retained)/len(active_2024):.1%}",                                                          # churn rate calculated as the complement of the retention rate
        f"{repeat_rate:.1%}",                                                                               # repeat purchase rate calculated as the proportion of customers with 2 or more orders
        f"{(customer_analytics['Segment']=='Champions').mean():.1%}",                                       # champions share calculated as the proportion of customers in the Champions segment
        f"{(customer_analytics['Segment']=='At Risk').mean():.1%}",                                         # at-risk share calculated as the proportion of customers in the At-Risk segment
        f"€{customer_analytics['CLV_Estimate'].mean():,.0f}",                                               # average CLV estimate across all customers
        f"€{customer_analytics[customer_analytics['Segment']=='Champions']['CLV_Estimate'].mean():,.0f}",   # average CLV estimate for customers in the Champions segment
    ],
})

# save the kpis marketing to parquet for future use
kpis_marketing.to_parquet("../data/curated/kpis_marketing.parquet", index=False)"""

#### Save outputs

Team 4 – Customer Segments
Business Question
Which customer segments create the most value?
Possible Topics
• Sales by segment
• Profit by segment
• Purchasing behaviour
Final Recommendation
Which segments should receive greater marketing attention?

df